# Create CelebA Hair Rich Dataset

In [1]:
from pathlib import Path
import sys

def find_project_root(start: Path | None = None) -> Path:
    current = (start or Path.cwd()).resolve()
    for candidate in [current, *current.parents]:
        if (candidate / 'backend').exists() and (candidate / 'notebooks').exists():
            return candidate
    raise RuntimeError('Could not locate project root.')

PROJECT_ROOT = find_project_root()
BACKEND_ROOT = PROJECT_ROOT / 'backend'
if str(BACKEND_ROOT) not in sys.path:
    sys.path.append(str(BACKEND_ROOT))

PROJECT_ROOT

WindowsPath('D:/Projects/Personal Projects/Hairstyle Recommender Live Tryon')

In [2]:
import json
from pathlib import Path

import pandas as pd

from systems.static_auto_tryon.auto_app.ml.celeba_hair_rich import build_celeba_seed_table, build_hair_rich_record, write_jsonl

RAW_CELEBA_ROOT = BACKEND_ROOT / 'data' / 'raw' / 'celeba'
DATASET_ROOT = BACKEND_ROOT / 'data' / 'datasets' / 'celeba_hair_rich'
MASK_OUTPUT_DIR = DATASET_ROOT / 'segmentation_masks'
TRAIN_PATH = DATASET_ROOT / 'train.jsonl'
VAL_PATH = DATASET_ROOT / 'val.jsonl'
TEST_PATH = DATASET_ROOT / 'test.jsonl'
SUMMARY_PATH = DATASET_ROOT / 'summary.json'
SUBSET_MANIFEST_PATH = DATASET_ROOT / 'lightweight_subset.jsonl'

In [3]:
seed_table = build_celeba_seed_table(RAW_CELEBA_ROOT)
subset_records = [json.loads(line) for line in SUBSET_MANIFEST_PATH.read_text(encoding='utf-8').splitlines() if line.strip()]
subset_by_id = {record['image_id']: record for record in subset_records}

print('Subset manifest rows:', len(subset_by_id))
seed_table = seed_table[seed_table['image_id'].isin(subset_by_id.keys())].copy()
print('Matched seed rows:', len(seed_table))

Subset manifest rows: 6000
Matched seed rows: 6000


In [4]:
rich_records = []
missing_masks = []

for _, row in seed_table.iterrows():
    image_id = row['image_id']
    mask_path = MASK_OUTPUT_DIR / f"{Path(image_id).stem}_hair_mask.png"
    if not mask_path.exists():
        missing_masks.append(str(mask_path))
        continue
    rich_records.append(build_hair_rich_record(RAW_CELEBA_ROOT, row, mask_path))

print('Rich records:', len(rich_records))
print('Missing masks:', len(missing_masks))
rich_records[0] if rich_records else None

Rich records: 6000
Missing masks: 0


{'image_id': '000001.jpg',
 'image_path': 'D:\\Projects\\Personal Projects\\Hairstyle Recommender Live Tryon\\backend\\data\\raw\\celeba\\img_align_celeba\\img_align_celeba\\000001.jpg',
 'partition': 'train',
 'gender_label': 'female',
 'celeba_attributes': {'Male': -1,
  'Bald': -1,
  'Bangs': -1,
  'Black_Hair': -1,
  'Blond_Hair': -1,
  'Brown_Hair': 1,
  'Gray_Hair': -1,
  'Straight_Hair': 1,
  'Wavy_Hair': -1,
  'Receding_Hairline': -1,
  'Wearing_Hat': -1},
 'segmentation_mask_path': 'D:\\Projects\\Personal Projects\\Hairstyle Recommender Live Tryon\\backend\\data\\datasets\\celeba_hair_rich\\segmentation_masks\\000001_hair_mask.png',
 'hair_mask_stats': {'positive_pixels': 20230,
  'coverage_ratio': 0.521338,
  'bbox_x': 0,
  'bbox_y': 0,
  'bbox_width': 178,
  'bbox_height': 218}}

In [5]:
train_records = [record for record in rich_records if record['partition'] == 'train']
val_records = [record for record in rich_records if record['partition'] == 'val']
test_records = [record for record in rich_records if record['partition'] == 'test']

write_jsonl(train_records, TRAIN_PATH)
write_jsonl(val_records, VAL_PATH)
write_jsonl(test_records, TEST_PATH)

summary = {
    'total_records': len(rich_records),
    'train_records': len(train_records),
    'val_records': len(val_records),
    'test_records': len(test_records),
    'male_records': sum(1 for record in rich_records if record['gender_label'] == 'male'),
    'female_records': sum(1 for record in rich_records if record['gender_label'] == 'female'),
    'avg_mask_coverage': round(sum(record['hair_mask_stats']['coverage_ratio'] for record in rich_records) / max(len(rich_records), 1), 6),
}

SUMMARY_PATH.write_text(json.dumps(summary, indent=2), encoding='utf-8')
summary

{'total_records': 6000,
 'train_records': 4000,
 'val_records': 1000,
 'test_records': 1000,
 'male_records': 2186,
 'female_records': 3814,
 'avg_mask_coverage': 0.273137}

In [6]:
pd.DataFrame(rich_records).head() if rich_records else pd.DataFrame()

,image_id,image_path,partition,gender_label,celeba_attributes,segmentation_mask_path,hair_mask_stats
0,000001.jpg,D:\Projects\Personal Projects\Hairstyle Recomm...,train,female,"{'Male': -1, 'Bald': -1, 'Bangs': -1, 'Black_H...",D:\Projects\Personal Projects\Hairstyle Recomm...,"{'positive_pixels': 20230, 'coverage_ratio': 0..."
1,000002.jpg,D:\Projects\Personal Projects\Hairstyle Recomm...,train,female,"{'Male': -1, 'Bald': -1, 'Bangs': -1, 'Black_H...",D:\Projects\Personal Projects\Hairstyle Recomm...,"{'positive_pixels': 10858, 'coverage_ratio': 0..."
2,000003.jpg,D:\Projects\Personal Projects\Hairstyle Recomm...,train,male,"{'Male': 1, 'Bald': -1, 'Bangs': -1, 'Black_Ha...",D:\Projects\Personal Projects\Hairstyle Recomm...,"{'positive_pixels': 3123, 'coverage_ratio': 0...."
3,000004.jpg,D:\Projects\Personal Projects\Hairstyle Recomm...,train,female,"{'Male': -1, 'Bald': -1, 'Bangs': -1, 'Black_H...",D:\Projects\Personal Projects\Hairstyle Recomm...,"{'positive_pixels': 24612, 'coverage_ratio': 0..."
4,000006.jpg,D:\Projects\Personal Projects\Hairstyle Recomm...,train,female,"{'Male': -1, 'Bald': -1, 'Bangs': -1, 'Black_H...",D:\Projects\Personal Projects\Hairstyle Recomm...,"{'positive_pixels': 15657, 'coverage_ratio': 0..."
